In [1]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from sklearn.metrics import f1_score, matthews_corrcoef
from scipy.stats import friedmanchisquare, wilcoxon

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 5)

In [2]:
FILES = {
    "late_fusion": Path(r"C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared\08_results_late_fusion_text_speech_eeg_narrative\late_fusion_subject_predictions.csv"),
    "eeg": Path(r"C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared\02_results_eeg_narrative_paper_folds_unimodal\eeg_subject_predictions.csv"),
    "early_fusion": Path(r"C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared\05_results_text_speech_eeg_conversation_level_svm\subject_predictions_global.csv"),
    "text_speech": Path(r"C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared\03_results_text_speech_xgboost_paper\text_speech_subject_predictions_global.csv"),
}

OUTPUT_DIR = FILES["late_fusion"].parent / "statistical_comparison_F1_MCC"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_BOOTSTRAPS_PER_FOLD = 100
RANDOM_STATE = 13
POS_LABEL = 1

In [3]:
def read_csv_checked(path):
    if not path.exists():
        raise FileNotFoundError(f"No se encuentra el archivo: {path}")
    return pd.read_csv(path)


late = read_csv_checked(FILES["late_fusion"])
eeg = read_csv_checked(FILES["eeg"])
early = read_csv_checked(FILES["early_fusion"])
text_speech = read_csv_checked(FILES["text_speech"])

print("late_fusion:", late.shape)
print("eeg:", eeg.shape)
print("early_fusion:", early.shape)
print("text_speech:", text_speech.shape)

late_fusion: (94, 8)
eeg: (94, 5)
early_fusion: (94, 5)
text_speech: (101, 4)


In [4]:
common_subjects = set(late["subject_id"])
common_subjects &= set(eeg["subject_id"])
common_subjects &= set(early["subject_id"])
common_subjects &= set(text_speech["subject_id"])

print("Sujetos comunes:", len(common_subjects))

late = late[late["subject_id"].isin(common_subjects)].copy()
eeg = eeg[eeg["subject_id"].isin(common_subjects)].copy()
early = early[early["subject_id"].isin(common_subjects)].copy()
text_speech = text_speech[text_speech["subject_id"].isin(common_subjects)].copy()

Sujetos comunes: 94


In [ ]:
preds = late[["subject_id", "label", "outer_fold", "pred_fusion"]].copy()
preds = preds.rename(columns={"pred_fusion": "pred_late_fusion"})

preds = preds.merge(
    text_speech[["subject_id", "label", "y_pred"]].rename(columns={
        "label": "label_text_speech",
        "y_pred": "pred_text_speech",
    }),
    on="subject_id",
    how="inner",
)

preds = preds.merge(
    eeg[["subject_id", "label", "outer_fold", "y_pred"]].rename(columns={
        "label": "label_eeg",
        "outer_fold": "outer_fold_eeg",
        "y_pred": "pred_eeg",
    }),
    on="subject_id",
    how="inner",
)

preds = preds.merge(
    early[["subject_id", "label", "outer_fold", "pred"]].rename(columns={
        "label": "label_early_fusion",
        "outer_fold": "outer_fold_early_fusion",
        "pred": "pred_early_fusion",
    }),
    on="subject_id",
    how="inner",
)

# Comprobaciones 
assert (preds["label"] == preds["label_text_speech"]).all(), "Labels no coinciden con Text+Speech"
assert (preds["label"] == preds["label_eeg"]).all(), "Labels no coinciden con EEG"
assert (preds["label"] == preds["label_early_fusion"]).all(), "Labels no coinciden con Early Fusion"
assert (preds["outer_fold"] == preds["outer_fold_eeg"]).all(), "Folds no coinciden con EEG"
assert (preds["outer_fold"] == preds["outer_fold_early_fusion"]).all(), "Folds no coinciden con Early Fusion"

preds = preds[[
    "subject_id", "label", "outer_fold",
    "pred_text_speech", "pred_eeg", "pred_early_fusion", "pred_late_fusion",
]].copy()

MODEL_PRED_COLS = {
    "Text+Speech": "pred_text_speech",
    "EEG": "pred_eeg",
    "Early Fusion": "pred_early_fusion",
    "Late Fusion": "pred_late_fusion",
}

print("Sujetos usados:", preds["subject_id"].nunique())
print("Folds:", sorted(preds["outer_fold"].unique()))
display(preds.head())

Sujetos usados: 94
Folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


,subject_id,label,outer_fold,pred_text_speech,pred_eeg,pred_early_fusion,pred_late_fusion
0,USER_01_CB,0,1,0,0,0,0
1,USER_01_CB2,0,3,0,0,0,0
2,USER_02_CB,0,5,0,0,0,0
3,USER_02_CB2,0,3,0,0,0,0
4,USER_03_CB,0,2,0,0,1,0


Funciones

In [6]:
def compute_metric(y_true, y_pred, metric):
    if metric == "F1":
        return f1_score(y_true, y_pred, pos_label=POS_LABEL, zero_division=0)
    if metric == "MCC":
        return matthews_corrcoef(y_true, y_pred)
    raise ValueError(f"Métrica no reconocida: {metric}")


METRICS = ["F1", "MCC"]

In [7]:
rows = []

for metric in METRICS:
    for fold in sorted(preds["outer_fold"].unique()):
        fold_df = preds[preds["outer_fold"] == fold]
        for model_name, pred_col in MODEL_PRED_COLS.items():
            rows.append({
                "metric": metric,
                "outer_fold": fold,
                "model": model_name,
                "n_subjects": len(fold_df),
                "value": compute_metric(fold_df["label"], fold_df[pred_col], metric),
            })

metrics_by_fold = pd.DataFrame(rows)
display(metrics_by_fold)

summary_by_fold = (
    metrics_by_fold
    .groupby(["metric", "model"])["value"]
    .agg(["mean", "std"])
    .reset_index()
)

display(summary_by_fold)
summary_by_fold.to_csv(OUTPUT_DIR / "summary_by_fold_F1_MCC.csv", index=False)

,metric,outer_fold,model,n_subjects,value
0,F1,1,Text+Speech,20,0.57143
1,F1,1,EEG,20,0.70588
2,F1,1,Early Fusion,20,0.73684
3,F1,1,Late Fusion,20,0.62500
4,F1,2,Text+Speech,18,0.57143
5,F1,2,EEG,18,0.66667
6,F1,2,Early Fusion,18,0.53333
7,F1,2,Late Fusion,18,0.62500
8,F1,3,Text+Speech,18,0.66667
9,F1,3,EEG,18,0.70588


,metric,model,mean,std
0,F1,EEG,0.68030,0.18642
1,F1,Early Fusion,0.66777,0.16432
2,F1,Late Fusion,0.71667,0.10785
3,F1,Text+Speech,0.63498,0.07532
4,MCC,EEG,0.52365,0.21289
5,MCC,Early Fusion,0.44191,0.25752
6,MCC,Late Fusion,0.57605,0.21315
7,MCC,Text+Speech,0.44673,0.09994


Bootstrapping

remuestreos dentro de cada outer fold.  
Con 5 folds y 100 bootstraps por fold

In [8]:
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_rows = []

for metric in METRICS:
    for fold in sorted(preds["outer_fold"].unique()):
        fold_df = preds[preds["outer_fold"] == fold].reset_index(drop=True)
        n = len(fold_df)
        
        for b in range(1, N_BOOTSTRAPS_PER_FOLD + 1):
            sample_idx = rng.choice(fold_df.index, size=n, replace=True)
            boot_df = fold_df.loc[sample_idx]
            
            for model_name, pred_col in MODEL_PRED_COLS.items():
                bootstrap_rows.append({
                    "metric": metric,
                    "outer_fold": fold,
                    "bootstrap": b,
                    "model": model_name,
                    "value": compute_metric(boot_df["label"], boot_df[pred_col], metric),
                })

bootstrap_long = pd.DataFrame(bootstrap_rows)
bootstrap_wide = bootstrap_long.pivot_table(
    index=["metric", "outer_fold", "bootstrap"],
    columns="model",
    values="value",
).reset_index()

print("Estimaciones por métrica:", bootstrap_wide.groupby("metric").size().to_dict())
display(bootstrap_wide.head())

bootstrap_long.to_csv(OUTPUT_DIR / "bootstrap_long_F1_MCC.csv", index=False)
bootstrap_wide.to_csv(OUTPUT_DIR / "bootstrap_wide_F1_MCC.csv", index=False)

Estimaciones por métrica: {'F1': 500, 'MCC': 500}


model,metric,outer_fold,bootstrap,EEG,Early Fusion,Late Fusion,Text+Speech
0,F1,1,1,0.46154,0.66667,0.00000,0.00000
1,F1,1,2,0.78261,0.84615,0.52632,0.47059
2,F1,1,3,0.84211,0.88889,0.84211,0.87500
3,F1,1,4,0.92308,0.82353,0.92308,0.83333
4,F1,1,5,0.57143,0.64000,0.50000,0.44444


 Test de Friedman



In [9]:
friedman_rows = []
model_order = list(MODEL_PRED_COLS.keys())

for metric in METRICS:
    data = bootstrap_wide[bootstrap_wide["metric"] == metric]
    stat, p_value = friedmanchisquare(*[data[model].values for model in model_order])
    
    friedman_rows.append({
        "metric": metric,
        "test": "Friedman",
        "n_models": len(model_order),
        "n_estimates_per_model": len(data),
        "statistic": stat,
        "p_value": p_value,
        "significant_0.05": p_value < 0.05,
    })

friedman_results = pd.DataFrame(friedman_rows)
display(friedman_results)

friedman_results.to_csv(OUTPUT_DIR / "friedman_results_F1_MCC.csv", index=False)

,metric,test,n_models,n_estimates_per_model,statistic,p_value,significant_0.05
0,F1,Friedman,4,500,79.72293,3.51939e-17,True
1,MCC,Friedman,4,500,81.27556,1.63458e-17,True


Comparaciones post-hoc: Wilcoxon + Bonferroni



In [10]:
posthoc_rows = []

for metric in METRICS:
    data = bootstrap_wide[bootstrap_wide["metric"] == metric]
    comparisons = list(combinations(model_order, 2))
    
    for model_a, model_b in comparisons:
        x = data[model_a].values
        y = data[model_b].values
        diff = x - y
        
        if np.allclose(diff, 0):
            stat = 0.0
            p_raw = 1.0
        else:
            stat, p_raw = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided")
        
        posthoc_rows.append({
            "metric": metric,
            "model_a": model_a,
            "model_b": model_b,
            "mean_a": x.mean(),
            "mean_b": y.mean(),
            "mean_difference_a_minus_b": x.mean() - y.mean(),
            "wilcoxon_statistic": stat,
            "p_value_raw": p_raw,
            "n_comparisons": len(comparisons),
            "p_value_bonferroni": min(p_raw * len(comparisons), 1.0),
        })

posthoc_results = pd.DataFrame(posthoc_rows)
posthoc_results["significant_0.05"] = posthoc_results["p_value_bonferroni"] < 0.05

display(posthoc_results)
posthoc_results.to_csv(OUTPUT_DIR / "posthoc_wilcoxon_bonferroni_F1_MCC.csv", index=False)

,metric,model_a,model_b,mean_a,mean_b,mean_difference_a_minus_b,wilcoxon_statistic,p_value_raw,n_comparisons,p_value_bonferroni,significant_0.05
0,F1,Text+Speech,EEG,0.62199,0.66043,-0.03844,45812.5,3.86412e-05,6,2.31847e-04,True
1,F1,Text+Speech,Early Fusion,0.62199,0.65640,-0.03440,38932.5,1.89612e-05,6,1.13767e-04,True
2,F1,Text+Speech,Late Fusion,0.62199,0.70780,-0.08581,24205.5,6.33665e-23,6,3.80199e-22,True
3,F1,EEG,Early Fusion,0.66043,0.65640,0.00403,46563.0,1.30443e-01,6,7.82657e-01,False
4,F1,EEG,Late Fusion,0.66043,0.70780,-0.04737,30244.0,1.71729e-04,6,1.03037e-03,True
5,F1,Early Fusion,Late Fusion,0.65640,0.70780,-0.05140,44291.5,5.06551e-04,6,3.03931e-03,True
6,MCC,Text+Speech,EEG,0.42829,0.50995,-0.08167,44092.0,3.05798e-07,6,1.83479e-06,True
7,MCC,Text+Speech,Early Fusion,0.42829,0.42943,-0.00114,50707.5,5.14366e-01,6,1.00000e+00,False
8,MCC,Text+Speech,Late Fusion,0.42829,0.56298,-0.13470,22205.0,2.08769e-27,6,1.25261e-26,True
9,MCC,EEG,Early Fusion,0.50995,0.42943,0.08053,36396.5,1.01737e-07,6,6.10421e-07,True


Resumen

In [11]:
summary_bootstrap = (
    bootstrap_long
    .groupby(["metric", "model"])["value"]
    .agg(["mean", "std"])
    .reset_index()
    .rename(columns={"mean": "bootstrap_mean", "std": "bootstrap_std"})
)

display(summary_bootstrap)
summary_bootstrap.to_csv(OUTPUT_DIR / "summary_bootstrap_F1_MCC.csv", index=False)

print("Archivos guardados en:")
print(OUTPUT_DIR)
print("- summary_by_fold_F1_MCC.csv")
print("- bootstrap_long_F1_MCC.csv")
print("- bootstrap_wide_F1_MCC.csv")
print("- friedman_results_F1_MCC.csv")
print("- posthoc_wilcoxon_bonferroni_F1_MCC.csv")
print("- summary_bootstrap_F1_MCC.csv")

,metric,model,bootstrap_mean,bootstrap_std
0,F1,EEG,0.66043,0.23220
1,F1,Early Fusion,0.65640,0.20765
2,F1,Late Fusion,0.70780,0.18675
3,F1,Text+Speech,0.62199,0.17615
4,MCC,EEG,0.50995,0.27435
5,MCC,Early Fusion,0.42943,0.32345
6,MCC,Late Fusion,0.56298,0.26546
7,MCC,Text+Speech,0.42829,0.22959


Archivos guardados en:
C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared\08_results_late_fusion_text_speech_eeg_narrative\statistical_comparison_F1_MCC
- summary_by_fold_F1_MCC.csv
- bootstrap_long_F1_MCC.csv
- bootstrap_wide_F1_MCC.csv
- friedman_results_F1_MCC.csv
- posthoc_wilcoxon_bonferroni_F1_MCC.csv
- summary_bootstrap_F1_MCC.csv
